<a href="https://colab.research.google.com/github/D2718281828nis/ML-MachineLearning-Graphs/blob/main/examples/hyperbolic_graph_basics.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Poincaré geometry at curvature $-c$ / Геометрия Пуанкаре с кривизной $-c$

**Curated example; verification pending.** The deterministic cells are designed for a fresh Google Colab CPU runtime, but a clean Colab run has not yet been recorded. This is not evidence that the root research notebooks run. / **Курируемый пример; проверка ожидается.** Детерминированные ячейки предназначены для чистого CPU runtime Google Colab, но такой запуск пока не зафиксирован. Это не доказывает работоспособность исследовательских notebook в корне.

**Goal / Цель.** Implement the ball for general $c>0$, distance, projection, Möbius operations, true geodesics, and origin exp/log maps. / Реализовать шар для общего $c>0$, расстояние, проекцию, операции Мёбиуса, настоящие геодезические и exp/log в начале координат.

**Runtime / Время:** under 1 minute / менее 1 минуты. **Dependencies / Зависимости:** NumPy and Matplotlib, preinstalled in Colab / предустановлены в Colab. **Theory / Теория:** [models and maps](../theory/hyperbolic-graphs/03-models-and-maps.md).

## Convention / Соглашение

We use sectional curvature $K=-c$, $c>0$, and / Используется секционная кривизна $K=-c$, $c>0$, и

$$\mathbb D_c^n=\{x\in\mathbb R^n:c\|x\|^2<1\},\qquad \lambda_x^c=\frac{2}{1-c\|x\|^2}.$$

With this scaling, $d_c(u,v)\to2\|u-v\|$ as $c\to0^+$. Every function below states its coordinate input/output. / При таком масштабе $d_c(u,v)\to2\|u-v\|$ при $c\to0^+$. Ниже указаны координатные входы и выходы.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

DTYPE = np.float64
EPS = 1e-12

def _check_c(c):
    c = float(c)
    if not np.isfinite(c) or c <= 0:
        raise ValueError("c must be finite and positive")
    return c

def inside_ball(x, c, margin=0.0):
    """Whether Euclidean coordinates x lie inside D_c^n."""
    c = _check_c(c)
    return np.linalg.norm(np.asarray(x, dtype=DTYPE)) < (1.0 - margin) / np.sqrt(c)

def project(x, c, eps=1e-7):
    """Radially project Euclidean coordinates into D_c^n."""
    c = _check_c(c)
    x = np.asarray(x, dtype=DTYPE)
    norm = np.linalg.norm(x)
    max_norm = (1.0 - eps) / np.sqrt(c)
    return x if norm <= max_norm or norm == 0.0 else x * (max_norm / norm)

def mobius_add(u, v, c):
    """Möbius addition u ⊕_c v for coordinates in D_c^n."""
    c = _check_c(c)
    u, v = np.asarray(u, dtype=DTYPE), np.asarray(v, dtype=DTYPE)
    if not inside_ball(u, c) or not inside_ball(v, c):
        raise ValueError("Möbius-add inputs must be inside D_c^n")
    uv, u2, v2 = u @ v, u @ u, v @ v
    denominator = 1 + 2*c*uv + c*c*u2*v2
    if denominator <= EPS:
        raise FloatingPointError("Möbius-add denominator is too small")
    return ((1 + 2*c*uv + c*v2)*u + (1 - c*u2)*v) / denominator

def mobius_scalar_mul(t, x, c):
    """Möbius scalar product t ⊗_c x in D_c^n."""
    c = _check_c(c)
    x = np.asarray(x, dtype=DTYPE)
    norm = np.linalg.norm(x)
    if norm == 0.0:
        return np.zeros_like(x)
    z = np.clip(np.sqrt(c)*norm, 0.0, 1.0 - EPS)
    return np.tanh(float(t)*np.arctanh(z)) * x / (np.sqrt(c)*norm)

def poincare_distance(u, v, c):
    """Geodesic distance between Euclidean coordinates u,v in D_c^n."""
    c = _check_c(c)
    delta = mobius_add(-np.asarray(u, dtype=DTYPE), v, c)
    z = np.clip(np.sqrt(c)*np.linalg.norm(delta), 0.0, 1.0 - EPS)
    return 2.0*np.arctanh(z)/np.sqrt(c)

def expmap0(w, c):
    """exp_0^c: tangent coordinates T_0 D_c^n -> D_c^n."""
    c = _check_c(c)
    w = np.asarray(w, dtype=DTYPE)
    norm = np.linalg.norm(w)
    return np.zeros_like(w) if norm == 0.0 else np.tanh(np.sqrt(c)*norm)*w/(np.sqrt(c)*norm)

def logmap0(x, c):
    """log_0^c: D_c^n -> tangent coordinates T_0 D_c^n."""
    c = _check_c(c)
    x = np.asarray(x, dtype=DTYPE)
    if not inside_ball(x, c):
        raise ValueError("log-map input must be inside D_c^n")
    norm = np.linalg.norm(x)
    if norm == 0.0:
        return np.zeros_like(x)
    z = np.clip(np.sqrt(c)*norm, 0.0, 1.0 - EPS)
    return np.arctanh(z)*x/(np.sqrt(c)*norm)

def geodesic(u, v, c, samples=100):
    """Samples gamma(t)=u ⊕_c (t ⊗_c ((-u) ⊕_c v)), t in [0,1]."""
    relative = mobius_add(-np.asarray(u, dtype=DTYPE), v, c)
    return np.array([mobius_add(u, mobius_scalar_mul(t, relative, c), c)
                     for t in np.linspace(0.0, 1.0, samples)])


## Deterministic sanity tests / Детерминированные проверки

The tests cover identity, symmetry, positivity, exp/log round trip, projection, boundary growth, geodesic endpoints and constant-speed behavior, and the declared Euclidean limit. / Тесты проверяют тождество, симметрию, положительность, round-trip exp/log, проекцию, рост у границы, концы и постоянную скорость геодезической и заявленный евклидов предел.

In [ ]:
def poincare_distance(u, v, eps=1e-12):
    """Distance in the unit-curvature Poincaré ball."""
    u, v = np.asarray(u, dtype=float), np.asarray(v, dtype=float)
    u2, v2 = u @ u, v @ v
    if u2 >= 1 or v2 >= 1:
        raise ValueError("Points must be strictly inside the unit ball")
    argument = 1 + 2 * np.sum((u - v) ** 2) / ((1 - u2) * (1 - v2))
    return np.arccosh(max(1.0, argument - eps))

origin = np.zeros(2)
for radius in (0.2, 0.5, 0.8, 0.95):
    print(f"radius={radius:.2f}, distance from center={poincare_distance(origin, [radius, 0]):.3f}")

assert np.isclose(poincare_distance(origin, origin), 0.0)
assert poincare_distance(origin, [0.8, 0]) > poincare_distance(origin, [0.5, 0])


## True Poincaré geodesics / Настоящие геодезические Пуанкаре

Edges below use Möbius geodesic interpolation, not straight visual-aid chords. In this two-dimensional conformal model, non-diameter geodesics appear as boundary-orthogonal circular arcs. / Рёбра ниже построены интерполяцией Мёбиуса, а не вспомогательными прямыми хордами. В двумерной конформной модели геодезические вне диаметров выглядят как дуги, ортогональные границе.

In [ ]:
levels = {
    "root": (0, 0.0, 0.0),
    "A": (1, 0.45, 0.0), "B": (1, 0.45, np.pi),
    "A1": (2, 0.78, -0.35), "A2": (2, 0.78, 0.35),
    "B1": (2, 0.78, np.pi - 0.35), "B2": (2, 0.78, np.pi + 0.35),
}
points = {name: np.array([radius * np.cos(angle), radius * np.sin(angle)])
          for name, (_, radius, angle) in levels.items()}
edges = [("root", "A"), ("root", "B"), ("A", "A1"), ("A", "A2"),
         ("B", "B1"), ("B", "B2")]

fig, ax = plt.subplots(figsize=(7, 7))
ax.add_patch(plt.Circle((0, 0), 1, fill=False, linewidth=2, color="black"))
for source, target in edges:
    # Chords aid readability; true Poincaré geodesics are generally circular arcs.
    xy = np.vstack([points[source], points[target]])
    ax.plot(xy[:, 0], xy[:, 1], color="#888888", zorder=1)
for name, point in points.items():
    ax.scatter(*point, s=350, color="#3568b8", zorder=2)
    ax.text(*point, name, ha="center", va="center", color="white", zorder=3)
ax.set(xlim=(-1.05, 1.05), ylim=(-1.05, 1.05), aspect="equal",
       title="A hierarchy inside the Poincaré disk")
ax.axis("off")
plt.show()

for source, target in edges:
    print(f"{source:>4} → {target:<2}: {poincare_distance(points[source], points[target]):.3f}")
assert all(np.linalg.norm(point) < 1 for point in points.values())


## Interpretation / Интерпретация

Euclidean radii are bounded by 1, but hyperbolic distance to the boundary is unbounded. The straight edge chords above are visual aids, not generally geodesics. / Евклидовы радиусы ограничены единицей, но гиперболическое расстояние до границы не ограничено. Прямые рёбра на рисунке — визуальная подсказка, а не геодезические.

Continue with the [bilingual theory](../theory/hyperbolic-graphs/README.md) / Продолжите с [двуязычной теорией](../theory/hyperbolic-graphs/README.md).